# NLI-pretrained classifier (DeBERTa-v3-large) against BiomedBERT

`scripts/train_nli.py`: an NLI model trained on about a million support/contradict/neutral pairs
(`MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli`) reads premise = our reranker's
evidence (best-ranked first) and hypothesis = the citing sentence, and its own NLI head is kept:
entailment = ACCURATE, contradiction = NOT ACCURATE, neutral = NOT ENOUGH INFO. Fine-tuned 3
epochs. Why: `results/diagnosis_v1.md`.

It is compared with BiomedBERT under **the same protocol**: the same 3-fold cross-validation over
the training articles (BiomedBERT, no context: macro-F1 0.428, accuracy 0.643, `classifier_v4.md`).

- **Phase 1** (about 70 minutes): 3-fold cross-validation, plus the zero-shot score on dev.
- **Phase 2** (about 1.5 hours): 3 seeds on all of train, predicting test. **It runs only if
  Phase 1 beats BiomedBERT**, otherwise the notebook stops there and saves the time.

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all, and allow
Google Drive access. Finished runs are saved to `MyDrive/CiteAudit_runs/v5`; if Colab disconnects,
Run all again and they are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RES = '/content/drive/MyDrive/CiteAudit_runs/v5'
import os; os.makedirs(RES, exist_ok=True)

%cd /content
!rm -rf CiteAudit && git clone -q https://github.com/SapnilPatel/CiteAudit.git
%cd /content/CiteAudit
!git log -1 --oneline
!pip -q install "transformers>=4.44" "tokenizers>=0.19" sentencepiece protobuf

import subprocess, json, statistics
def run(cmd):
    noise = ('UNEXPECTED', 'MISSING', 'LOAD REPORT', 'Key ', '----', 'Notes', '- UNEXP', '- MISS',
             'Loading weights', 'Warning', 'warn(')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not any(n in line for n in noise):
            print(line, end='')
    if p.wait():
        raise RuntimeError(f'failed ({p.returncode}): {cmd}')

T = 'data_train/target'
def train(name, seed, extra):
    out = f'{RES}/{name}'
    if os.path.exists(f'{out}/config.json'):
        print(f'{name}: already done'); return
    run(f'python scripts/train_nli.py --train {T}/ours_20r_train_cv --dev {T}/ours_20r '
        f'--epochs 3 --batch 2 --accum 4 --seed {seed} --amp '
        f'--out checkpoints/{name} --pred-dir {out}.part {extra}')
    run(f'rm -rf checkpoints/{name}')
    os.rename(f'{out}.part', out)

LABELS = ['ACCURATE', 'NOT_ACCURATE', 'NOT ENOUGH INFO']
def scores(rows):
    f1 = []
    for c in LABELS:
        tp = sum(r['gold'] == c and r['prediction'] == c for r in rows)
        fp = sum(r['gold'] != c and r['prediction'] == c for r in rows)
        fn = sum(r['gold'] == c and r['prediction'] != c for r in rows)
        f1.append(2 * tp / (2 * tp + fp + fn) if tp else 0.)
    return [sum(r['gold'] == r['prediction'] for r in rows) / len(rows), sum(f1) / 3] + f1
read = lambda p: [json.loads(l) for l in open(p)]
COLS = ['accuracy', 'macro-F1', 'ACC F1', 'N_ACC F1', 'NEI F1']

## Phase 1: 3-fold cross-validation over train

In [ ]:
for k in range(3):
    train(f'cv_nli_f{k}', 0, f'--folds 3 --fold {k}')

oof = sum((read(f'{RES}/cv_nli_f{k}/oof.jsonl') for k in range(3)), [])
nli = scores(oof)
BASE = [0.643, 0.428, 0.758, 0.454, 0.072]   # BiomedBERT, same folds (classifier_v4.md)
print(f'{"cross-validated, 2141 citations":32s} ' + ' '.join(f'{c:>9s}' for c in COLS))
print(f'{"BiomedBERT (no context)":32s} ' + ' '.join(f'{x:9.3f}' for x in BASE))
print(f'{"NLI DeBERTa-v3-large":32s} ' + ' '.join(f'{x:9.3f}' for x in nli))
print('per-fold macro-F1:', [round(scores(read(f'{RES}/cv_nli_f{k}/oof.jsonl'))[1], 3) for k in range(3)])
zs = json.load(open(f'{RES}/cv_nli_f0/config.json'))['log'][0]
print(f'zero-shot on dev (no training): macro-F1 {zs["dev_macro_f1"]:.3f}, accuracy {zs["dev_accuracy"]:.3f}')
GO = nli[1] > BASE[1]
json.dump({'nli_cv': nli, 'biomedbert_cv': BASE, 'phase2': GO}, open(f'{RES}/cv_summary.json', 'w'), indent=2)
print('\nPhase 2 will run.' if GO else '\nThe NLI model does not beat BiomedBERT; Phase 2 is skipped.')

## Phase 2: 3 seeds on all of train, predicting test (only if Phase 1 won)

In [ ]:
if GO:
    for seed in (0, 1, 2):
        train(f'final_nli_s{seed}', seed, f'--predict test={T}/ours_20r_test')
    runs = [scores(read(f'{RES}/final_nli_s{s}/ours_20r_test.jsonl')) for s in (0, 1, 2)]
    print('test, mean ± sd over 3 seeds')
    for name, xs in zip(COLS, zip(*runs)):
        print(f'  {name:9s} {statistics.mean(xs):.3f} ± {statistics.stdev(xs):.3f}')
    print('BiomedBERT on test (classifier_v3.md, E3): accuracy 0.662, macro-F1 0.489')

## Download

Zips `MyDrive/CiteAudit_runs/v5` (predictions and logs, no weights). Put `v5.zip` in the repository
root and tell Claude.

In [ ]:
!cd /content/drive/MyDrive/CiteAudit_runs && zip -qr /content/v5.zip v5 -x '*.part/*'
from google.colab import files
files.download('/content/v5.zip')